# Week 7：模型对比——从线性模型到集成模型

目标：在完全相同的数据划分和 5 折交叉验证规则下，对比四种分类模型。

本节要形成的工程习惯：先选一个简单、可解释的基线，再用更复杂的模型挑战它；最终依据验证结果、稳定性、成本和可解释性选择候选模型。

## 模型清单

- **Logistic Regression**：线性基线，速度快、可解释。
- **Random Forest**：Bagging；多棵相对独立的树投票，主要降低不稳定性。
- **Gradient Boosting**：Boosting；后面的树依次修正前面的错误。
- **XGBoost**：带正则化、采样与工程优化的 Gradient Boosting 实现。

这里使用 accuracy 作为统一比较指标。真实项目还需要依照业务目标补充 recall、F1、AUC 等指标。

In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

In [2]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

# 最终测试集仍然封存；本节只在开发数据上完成模型比较。
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
models = {
    # 线性模型依赖特征尺度，因此把标准化放进 Pipeline，保证每折只学习该折训练数据的统计量。
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    # Bagging：树之间并行训练并投票。
    'Random Forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=1, random_state=42, n_jobs=1
    ),
    # 传统 Gradient Boosting：顺序叠加树修正错误。
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=2, random_state=42
    ),
    # XGBoost：同样是 Boosting，但加入采样和 L2 正则化。
    'XGBoost': XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        eval_metric='logloss', random_state=42, n_jobs=1
    ),
}

In [4]:
rows = []

for name, model in models.items():
    # 这里会做 5 次临时训练，每次用 4 折训练、1 折验证。
    scores = cross_validate(
        model, X_train, y_train, cv=cv, scoring='accuracy',
        return_train_score=True, n_jobs=-1,
    )
    rows.append({
        'model': name,
        'train_mean': scores['train_score'].mean(),
        'validation_mean': scores['test_score'].mean(),
        'validation_std': scores['test_score'].std(),
    })

comparison = pd.DataFrame(rows).sort_values('validation_mean', ascending=False)
comparison['gap'] = comparison['train_mean'] - comparison['validation_mean']
display(comparison)

,model,train_mean,validation_mean,validation_std,gap
0,Logistic Regression,0.989011,0.978022,0.009829,0.010989
3,XGBoost,1.000000,0.973626,0.016447,0.026374
1,Random Forest,1.000000,0.964835,0.017582,0.035165
2,Gradient Boosting,1.000000,0.962637,0.013187,0.037363


## 如何给出工程结论

1. 先看 `validation_mean`：谁在未参与训练的折上平均表现更好。
2. 再看 `validation_std`：谁在不同折上的波动更小。
3. 看 `gap`：训练与验证的差距过大时要警惕过拟合。
4. 若分数很接近，优先考虑更简单、更快或更可解释的模型。

思考题：如果 XGBoost 只比 Logistic Regression 高 0.002，但训练和部署复杂许多，你会立刻选 XGBoost 吗？为什么？